# Pléyade · Eos · modelo voz→cara (GPU)

1. Rasgos wav2vec2 del audio de los discursos de Obama (dominio público) → Drive.
2. Entrenamiento del transformer voz→cara con Obama + TalkingHead-1KH.

Requiere un entorno **GPU (T4 gratuita)** y Drive montado. Reanudable.

In [ ]:
EXTRAER_PY = "\"\"\"Rasgos de voz para el modelo voz→cara: wav2vec2-base (preentrenado, Apache-2.0) sobre el\naudio de cada vídeo medido, en la misma rejilla de 25 fps que su .npz de medidas.\n\nPor vídeo: audio a 16 kHz mono (ffmpeg) → wav2vec2-base → media de las capas ocultas 6-9\n(las que mejor codifican fonemas y prosodia) → 50 Hz a 25 Hz promediando pares → float16.\nGuarda <salida>/<nombre>.npz con `voz` [T, 768] y `t` [T]. Reanudable; trozos de 20 s para\nno agotar la memoria con vídeos largos (discursos de Obama).\n\nUso:  python extraer.py --medidas DIR_NPZ --videos DIR_MP4 --salida DIR [--dispositivo cuda]\nEmpareja <nombre>.npz con <nombre>.mp4 (busca recursivamente en --videos).\n\"\"\"\nimport argparse\nimport glob\nimport os\nimport subprocess\n\nimport numpy as np\nimport torch\n\n\ndef audio(mp4, ffmpeg):\n    r = subprocess.run([ffmpeg, '-loglevel', 'error', '-i', mp4, '-ac', '1', '-ar', '16000', '-f', 's16le', '-'],\n                       capture_output=True, check=True)\n    return np.frombuffer(r.stdout, np.int16).astype(np.float32) / 32768\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--medidas', required=True)\n    ap.add_argument('--videos', required=True)\n    ap.add_argument('--salida', required=True)\n    ap.add_argument('--dispositivo', default='cuda' if torch.cuda.is_available() else 'cpu')\n    a = ap.parse_args()\n    from transformers import Wav2Vec2Model\n    try:\n        import imageio_ffmpeg\n        ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()\n    except ImportError:\n        ffmpeg = 'ffmpeg'\n    os.makedirs(a.salida, exist_ok=True)\n    red = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(a.dispositivo).eval()\n    videos = {os.path.splitext(os.path.basename(v))[0]: v\n              for v in glob.glob(os.path.join(a.videos, '**', '*.mp4'), recursive=True)}\n    hechos = 0\n    for npz in sorted(glob.glob(os.path.join(a.medidas, '**', '*.npz'), recursive=True)):\n        nombre = os.path.splitext(os.path.basename(npz))[0]\n        dest = os.path.join(a.salida, nombre + '.npz')\n        if os.path.exists(dest) or nombre not in videos:\n            continue\n        t = np.load(npz)['t']\n        try:\n            x = audio(videos[nombre], ffmpeg)\n        except Exception as e:\n            print('sin audio', nombre, e)\n            continue\n        trozos = []\n        with torch.no_grad():\n            for i in range(0, len(x), 16000 * 20):\n                seg = torch.tensor(x[i:i + 16000 * 20])[None].to(a.dispositivo)\n                if seg.shape[1] < 400:\n                    break\n                h = red(seg, output_hidden_states=True).hidden_states\n                trozos.append(torch.stack(h[6:10]).mean(0)[0].cpu().numpy())    # [T50, 768]\n        f50 = np.concatenate(trozos)\n        f25 = f50[: len(f50) // 2 * 2].reshape(-1, 2, 768).mean(1)                # 25 Hz\n        g = np.arange(len(f25)) / 25.0\n        idx = np.clip(np.searchsorted(g, t), 0, len(f25) - 1)                     # a la rejilla del .npz\n        np.savez_compressed(dest, voz=f25[idx].astype(np.float16), t=t)\n        hechos += 1\n        if hechos % 20 == 0:\n            print(hechos, nombre, flush=True)\n    print('listo', hechos)\n\n\nif __name__ == '__main__':\n    main()\n"
ENTRENAR_PY = "\"\"\"Modelo voz→cara: de los rasgos wav2vec2 (extraer.py) a los 52 blendshapes ARKit y la pose\nde la cabeza, en la rejilla de fotogramas de cada vídeo (≈25-30 fps). Transformer pequeño no causal sobre ventanas de 10 s.\n\nObjetivo por clip: blendshapes menos su media en el clip (así no aprende la cara en reposo de\ncada hablante, sí su movimiento), escalados por la sd global; pose en grados menos su media.\nPérdida: MSE en fotogramas con cara + 0,5·MSE de la velocidad (movimiento suave).\nValidación por vídeo: la carpeta val de 1KH y 1 de cada 8 vídeos de Obama y Miralles.\n\nUso: python entrenar.py --voz DIR_RASGOS --medidas DIR_NPZ --salida DIR [--epocas 30] [--dispositivo cuda]\nDeja modelo.pt, normalizacion.json e informe.json (r de Pearson por grupo y por blendshape).\n\"\"\"\nimport argparse\nimport glob\nimport json\nimport os\nimport zlib\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nGRUPOS = {\n    'boca': ['jawOpen', 'mouthClose', 'mouthFunnel', 'mouthPucker', 'mouthPressLeft', 'mouthPressRight',\n             'mouthRollLower', 'mouthRollUpper', 'mouthLowerDownLeft', 'mouthLowerDownRight',\n             'mouthUpperUpLeft', 'mouthUpperUpRight', 'mouthStretchLeft', 'mouthStretchRight'],\n    'sonrisa': ['mouthSmileLeft', 'mouthSmileRight', 'cheekSquintLeft', 'cheekSquintRight'],\n    'ojos': ['eyeBlinkLeft', 'eyeBlinkRight', 'eyeSquintLeft', 'eyeSquintRight', 'eyeWideLeft', 'eyeWideRight'],\n    'cejas': ['browInnerUp', 'browOuterUpLeft', 'browOuterUpRight', 'browDownLeft', 'browDownRight'],\n}\n\n\ndef pares(dir_voz, dir_med):\n    med = {os.path.splitext(os.path.basename(f))[0]: f\n           for f in glob.glob(os.path.join(dir_med, '**', '*.npz'), recursive=True)}\n    for v in sorted(glob.glob(os.path.join(dir_voz, '**', '*.npz'), recursive=True)):\n        n = os.path.splitext(os.path.basename(v))[0]\n        if n in med:\n            yield n, v, med[n]\n\n\ndef cargar(fv, fm):\n    v, m = np.load(fv), np.load(fm)\n    x = v['voz'].astype(np.float32)\n    ok = m['ok'].astype(bool)\n    if ok.mean() < 0.6 or len(x) < 75:\n        return None\n    bs = m['bs'].astype(np.float32)\n    pose = np.nan_to_num(m['pose'].astype(np.float32))\n    y = np.concatenate([bs, pose], 1)\n    y = y - y[ok].mean(0)\n    y[~ok] = 0\n    return x, y, ok, list(m['bs_nombres']), float(m['t'][-1] - m['t'][0])\n\n\nclass Red(nn.Module):\n    def __init__(self, ent=768, sal=55, d=256, capas=4, cabezas=4):\n        super().__init__()\n        self.entrada = nn.Sequential(nn.LayerNorm(ent), nn.Linear(ent, d))\n        self.conv = nn.Conv1d(d, d, 5, padding=2, groups=d)                    # posición relativa local\n        capa = nn.TransformerEncoderLayer(d, cabezas, 4 * d, dropout=0.1, batch_first=True, norm_first=True)\n        self.cuerpo = nn.TransformerEncoder(capa, capas)\n        self.salida = nn.Linear(d, sal)\n\n    def forward(self, x):\n        h = self.entrada(x)\n        h = h + self.conv(h.transpose(1, 2)).transpose(1, 2)\n        return self.salida(self.cuerpo(h))\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument('--voz', required=True)\n    ap.add_argument('--medidas', required=True)\n    ap.add_argument('--salida', required=True)\n    ap.add_argument('--epocas', type=int, default=30)\n    ap.add_argument('--dispositivo', default='cuda' if torch.cuda.is_available() else 'cpu')\n    a = ap.parse_args()\n    os.makedirs(a.salida, exist_ok=True)\n    todos = list(pares(a.voz, a.medidas))\n    # 1KH val solo tiene 3 clips: se reserva además 1 de cada 8 vídeos de YouTube de 1KH\n    # (por vídeo de origen, no por trozo, para que ningún hablante esté en los dos lados)\n    externo = lambda fm: any(k in fm for k in ('obama', 'casa_blanca', 'miralles'))\n    es_val = lambda i, n, fm: (os.sep + 'val' + os.sep in fm.replace('/', os.sep)) or \\\n        (externo(fm) and i % 8 == 0) or \\\n        (not externo(fm) and zlib.crc32(n.rsplit('_', 1)[0].encode()) % 8 == 0)\n    ent, val, nombres, horas = [], [], None, 0.0\n    for i, (n, fv, fm) in enumerate(todos):\n        r = cargar(fv, fm)\n        if r is None:\n            continue\n        nombres = r[3]\n        if es_val(i, n, fm):\n            val.append(r[:3])\n        else:\n            ent.append(r[:3]); horas += r[4] / 3600\n    Y = np.concatenate([y[ok] for _, y, ok in ent])\n    sd = Y.std(0) + 1e-3\n    print(f'entrenamiento {len(ent)} clips ({horas:.1f} h), validación {len(val)} clips', flush=True)\n\n    L = 250\n    def ventanas(datos):\n        X, Yv, M = [], [], []\n        for x, y, ok in datos:\n            for i in range(0, max(1, len(x) - L + 1), L // 2):\n                xs, ys, ms = x[i:i + L], y[i:i + L] / sd, ok[i:i + L]\n                if len(xs) < L:\n                    pad = L - len(xs)\n                    xs, ys, ms = np.pad(xs, ((0, pad), (0, 0))), np.pad(ys, ((0, pad), (0, 0))), np.pad(ms, (0, pad))\n                X.append(xs); Yv.append(ys); M.append(ms)\n        return torch.tensor(np.array(X)), torch.tensor(np.array(Yv), dtype=torch.float32), torch.tensor(np.array(M))\n    Xe, Ye, Me = ventanas(ent)\n    dev = a.dispositivo\n    red = Red(sal=Ye.shape[2]).to(dev)\n    opt = torch.optim.AdamW(red.parameters(), 3e-4, weight_decay=0.01)\n    pasos = a.epocas * ((len(Xe) + 31) // 32)\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=pasos)\n    for ep in range(a.epocas):\n        red.train(); perm = torch.randperm(len(Xe)); tot = 0\n        for i in range(0, len(perm), 32):\n            b = perm[i:i + 32]\n            x, y, m = Xe[b].to(dev), Ye[b].to(dev), Me[b].to(dev).unsqueeze(-1).float()\n            x = x + 0.05 * torch.randn_like(x)\n            p = red(x)\n            l1 = ((p - y) ** 2 * m).sum() / (m.sum() * y.shape[2])\n            dv = (p[:, 1:] - p[:, :-1]) - (y[:, 1:] - y[:, :-1])\n            l2 = (dv ** 2 * m[:, 1:] * m[:, :-1]).sum() / (m.sum() * y.shape[2])\n            loss = l1 + 0.5 * l2\n            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(red.parameters(), 1.0); opt.step(); sched.step()\n            tot += loss.item() * len(b)\n        print(f'época {ep + 1}: {tot / len(perm):.4f}', flush=True)\n\n    red.eval(); P, T = [], []\n    with torch.no_grad():\n        for x, y, ok in val:\n            p = red(torch.tensor(x)[None].to(dev))[0].cpu().numpy() * sd\n            P.append(p[ok]); T.append(y[ok])\n    P, T = np.concatenate(P), np.concatenate(T)\n    col = nombres + ['cabeza_guinada', 'cabeza_cabeceo', 'cabeza_alabeo']\n    r = {c: float(np.corrcoef(P[:, k], T[:, k])[0, 1]) if T[:, k].std() > 1e-6 else None for k, c in enumerate(col)}\n    grupos = {g: round(float(np.nanmean([r[c] for c in cs if r.get(c) is not None])), 3) for g, cs in GRUPOS.items()}\n    grupos['cabeza'] = round(float(np.mean([r[c] for c in col[-3:]])), 3)\n    print('r por grupo:', grupos, flush=True)\n    torch.save(red.state_dict(), os.path.join(a.salida, 'modelo.pt'))\n    json.dump({'columnas': col, 'sd': sd.tolist()}, open(os.path.join(a.salida, 'normalizacion.json'), 'w'))\n    json.dump({'clips_entrenamiento': len(ent), 'horas_entrenamiento': round(horas, 2), 'clips_validacion': len(val),\n               'r_grupos': grupos, 'r': {k: (round(v, 3) if v is not None else None) for k, v in r.items()}},\n              open(os.path.join(a.salida, 'informe.json'), 'w'), indent=1)\n\n\nif __name__ == '__main__':\n    main()\n"
# --- 1. Drive, dependencias y guiones
# En VS Code, drive.mount() falla: montar con Ctrl+Shift+P → «Colab: Mount Google Drive to Server...».
import os
if not os.path.isdir('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        raise SystemExit('Monta Drive con «Colab: Mount Google Drive to Server...» y vuelve a ejecutar') from e
!pip -q install imageio-ffmpeg
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA (cambiar el entorno a T4)')
CB = '/content/drive/MyDrive/literalevel_cara/casa_blanca'
V2C = '/content/drive/MyDrive/literalevel_cara/voz2cara'
os.makedirs(V2C, exist_ok=True)
open('/content/extraer.py', 'w', encoding='utf8').write(EXTRAER_PY)
open('/content/entrenar.py', 'w', encoding='utf8').write(ENTRENAR_PY)

In [ ]:
# --- 2. Rasgos de Obama (GPU, ~10-20 min; reanudable). Se guardan en local y en Drive
!mkdir -p /content/datos/voz/obama /content/datos/medidas/obama
!cp -n {V2C}/voz_obama/*.npz /content/datos/voz/obama/ 2>/dev/null; cp -n {CB}/medidas/*.npz /content/datos/medidas/obama/
!python -u /content/extraer.py --medidas /content/datos/medidas/obama --videos {CB}/videos --salida /content/datos/voz/obama --dispositivo cuda
!mkdir -p {V2C}/voz_obama && cp -n /content/datos/voz/obama/*.npz {V2C}/voz_obama/
!ls /content/datos/voz/obama | wc -l

In [ ]:
# --- 3. Rasgos y medidas de 1KH (subidos desde el PC como voz2cara/1kh.zip)
!cd /content/datos && unzip -q -n {V2C}/1kh.zip && ls voz medidas

In [ ]:
# --- 4. Entrenar (GPU). Validación: 1KH val + 1 de cada 8 discursos de Obama
!python -u /content/entrenar.py --voz /content/datos/voz --medidas /content/datos/medidas --salida {V2C}/modelo --epocas 30 --dispositivo cuda
!cat {V2C}/modelo/informe.json | head -n 12